# Empirical Justification of the Banpool Partitioning Protocol

**Purpose.** Quantify why the Banpool (unseen-word) protocol exposes a model's out-of-distribution
generalisation limits more sharply than the standard random split used by
Gaddy & Klein (2022) and Benster et al. (2024).

**Claim under test.** Word-frequency distributions in natural language cause frequent lexical
recurrence across random splits. A random split therefore leaks most of the evaluation lexicon
into training, masking true OOD limits. The Banpool protocol raises the token-level OOV rate
of the evaluation sets substantially.

**Design.** Prior sEMG-SSI work partitions the parallel data by randomly selecting a fixed
**100 validation** and **200 test** utterances, with the remainder as training. We reproduce that
protocol exactly (absolute counts, not ratios) as the baseline.

| Protocol | Train | Val | Test | Repetitions |
|---|---|---|---|---|
| Standard Random | remainder | 100 (fixed count) | 200 (fixed count) | 10 seeds, $\mathcal{S}=\{1,\dots,10\}$ |
| Banpool | pool $\mathcal{A}$ (fixed) | pool $\mathcal{B}$ 50% | pool $\mathcal{B}$ 50% | 10 permutations of pool $\mathcal{B}$ |
| Banpool (deployed) | pool $\mathcal{A}$ | pool $\mathcal{B}$ 50% | pool $\mathcal{B}$ 50% | `seed_split = 42` |

**Metric.** Token-level OOV rate: the percentage of *tokens* in a split whose type does not occur
anywhere in the training split's vocabulary. Reported separately for the word and token label
granularities, since the thesis trains CTC decoders at both.

$$\text{OOV}_{\text{rate}}(S) = 100 \cdot \frac{|\{t \in \text{tokens}(S) : t \notin V_{\text{train}}\}|}{|\text{tokens}(S)|}$$

Note this is a *token* rate (counts every occurrence), not a *type* rate and not an
*utterance* rate. All three are reported; the table in the thesis uses the token rate.

Because the two protocols yield evaluation sets of different sizes (300 vs. 450 utterances), the
OOV rate — a per-token proportion — is the quantity that makes them comparable.


In [ ]:
import sys, io, contextlib
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path().resolve().parents[1]  # .../Data Analysis
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from data.extract import load_dataset
from data.transform import transform_dataset, tokens_from_text # [CENSORED]
from data.split import split_dataset, split_dvss_unseen_protocol

BANLIST_PATH = ROOT / 'data' / 'banlist.txt'
DEPLOYED_SEED = 42          # seed_split used in the thesis experiments
N_REPS = 10                 # S = {1, ..., 10}

# Literature baseline: absolute counts, not ratios (Gaddy & Klein 2022; Benster et al. 2024)
N_VAL_BASELINE = 100
N_TEST_BASELINE = 200

# Banpool still partitions pool B 50/50 into val/test
VAL_RATIO, TEST_RATIO = 0.1, 0.1   # equal -> 50/50 of pool B
TRAIN_RATIO = 0.8                  # unused by the banpool path (pool A is the train set)

pd.options.display.float_format = '{:.2f}'.format
print('repo root:', ROOT)
print('banlist  :', BANLIST_PATH, '| exists:', BANLIST_PATH.exists())
print(f'baseline : {N_VAL_BASELINE} val / {N_TEST_BASELINE} test (absolute counts)')


## 1. Load $C_{1,\text{open}}$ (silent parallel subset)

We replicate the exact pipeline `train.py` executes: `load_dataset` → `transform_dataset`
(with `drop_empty_tokens=True`) → split. The transform step drops 6 utterances whose transcript
is a bare `"."` and which therefore yield an empty label sequence, taking 1,588 raw parallel
utterances to 1,582 analysable ones.

Transcripts are identical across the silent/voiced parallel modes, so the lexical statistics
computed here hold for both.


In [ ]:
df_raw = load_dataset(
    'DVSS', mode='silent', vocabulary_type='open_vocab',
    require_text=True, require_emg=True,
)
df_raw = df_raw[df_raw['group'] == 'silent_parallel_data'].reset_index(drop=True)
n_raw = len(df_raw)

# token_based is irrelevant for the drop filter (both granularities drop the same rows);
# we attach both token views ourselves below.
model_df, _, _ = transform_dataset(
    df_raw, drop_empty_tokens=True, token_based=False, load_emg_data=False,
)
model_df = model_df.sort_values('trial_id').reset_index(drop=True)  # determinism, as in train.py

model_df['words']  = model_df['text'].apply(tokens_from_text)
# [CENSORED / PROPRIETARY: Intermediate token target generation withheld]
model_df['tokens'] = model_df['text'].apply(lambda t: [f'T{i%39+1:02d}' for i in range(len(t.split()))])

print(f'raw parallel utterances      : {n_raw}')
print(f'analysable (after drop_empty): {len(model_df)}  (dropped {n_raw - len(model_df)})')
print(f'word types  |V| : {len({t for s in model_df["words"]  for t in s})}')
print(f'token types   : {len({t for s in model_df["phones"] for t in s})}')
print(f'word tokens     : {sum(len(s) for s in model_df["words"])}')


## 2. OOV metrics

`oov_token_rate` is the headline metric. `oov_type_rate` and `oov_utterance_rate` are reported
as supporting context — the utterance rate in particular shows that the Banpool protocol
guarantees *every* evaluation utterance contains at least one unseen word, which is the
protocol's defining property.


In [ ]:
def train_vocab(train_df: pd.DataFrame, col: str) -> set:
    """Set of token types occurring anywhere in the training split."""
    return {t for seq in train_df[col] for t in seq}


def oov_token_rate(train_df, eval_df, col: str) -> float:
    """% of token occurrences in eval_df whose type is absent from the train vocabulary."""
    vocab = train_vocab(train_df, col)
    tokens = [t for seq in eval_df[col] for t in seq]
    if not tokens:
        return float('nan')
    return 100.0 * sum(t not in vocab for t in tokens) / len(tokens)


def oov_type_rate(train_df, eval_df, col: str) -> float:
    """% of distinct token types in eval_df absent from the train vocabulary."""
    vocab = train_vocab(train_df, col)
    types = {t for seq in eval_df[col] for t in seq}
    if not types:
        return float('nan')
    return 100.0 * sum(t not in vocab for t in types) / len(types)


def oov_utterance_rate(train_df, eval_df, col: str) -> float:
    """% of eval utterances containing >= 1 token absent from the train vocabulary."""
    vocab = train_vocab(train_df, col)
    if len(eval_df) == 0:
        return float('nan')
    return 100.0 * np.mean([any(t not in vocab for t in seq) for seq in eval_df[col]])


def measure(train_df, val_df, test_df) -> dict:
    """All reported quantities for one concrete (train, val, test) partition."""
    return {
        'val_word_oov':   oov_token_rate(train_df, val_df,  'words'),
        'val_token_oov':   oov_token_rate(train_df, val_df,  'phones'),
        'test_word_oov':  oov_token_rate(train_df, test_df, 'words'),
        'test_token_oov':  oov_token_rate(train_df, test_df, 'phones'),
        'test_word_oov_type': oov_type_rate(train_df, test_df, 'words'),
        'test_word_oov_utt':  oov_utterance_rate(train_df, test_df, 'words'),
        'n_train': len(train_df), 'n_val': len(val_df), 'n_test': len(test_df),
    }


def split_absolute(df: pd.DataFrame, *, seed: int, n_val: int, n_test: int, shuffle: bool = True):
    """Random split by absolute counts: n_val validation, n_test test, remainder train.

    Mirrors data.split.split_dataset's RNG and slice order (train | val | test) so the two
    baselines differ only in how the split sizes are determined, not in how they are drawn.
    """
    n = len(df)
    if n_val + n_test > n:
        raise ValueError(f'n_val + n_test = {n_val + n_test} exceeds corpus size {n}')

    indices = np.arange(n)
    if shuffle:
        np.random.default_rng(seed).shuffle(indices)

    n_train = n - n_val - n_test
    train_idx = indices[:n_train]
    val_idx = indices[n_train:n_train + n_val]
    test_idx = indices[n_train + n_val:]

    return (
        df.iloc[train_idx].reset_index(drop=True),
        df.iloc[val_idx].reset_index(drop=True),
        df.iloc[test_idx].reset_index(drop=True),
    )


def _quiet_banpool(df, seed):
    """split_dvss_unseen_protocol prints progress; suppress it inside sweeps."""
    with contextlib.redirect_stdout(io.StringIO()):
        return split_dvss_unseen_protocol(
            df, banlist_path=BANLIST_PATH, seed=seed,
            train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO, test_ratio=TEST_RATIO,
            shuffle=True,
        )


## 3. Baseline: standard random split (100 val / 200 test) across 10 seeds

The literature default (Gaddy & Klein 2022; Benster et al. 2024): a fixed **100 validation** and
**200 test** utterances drawn uniformly at random, with the remaining 1,282 utterances used for
training. Sizes are absolute counts, not ratios. Each seed reshuffles the whole corpus, so
train/val/test all draw from the same lexical distribution and no unseen-word constraint applies.


In [ ]:
random_runs = []
for seed in range(1, N_REPS + 1):
    tr, va, te = split_absolute(
        model_df, seed=seed, n_val=N_VAL_BASELINE, n_test=N_TEST_BASELINE, shuffle=True,
    )
    random_runs.append({'seed': seed, **measure(tr, va, te)})

random_df = pd.DataFrame(random_runs)

# sanity: the baseline split sizes are exactly the literature's absolute counts
assert (random_df['n_val'] == N_VAL_BASELINE).all()
assert (random_df['n_test'] == N_TEST_BASELINE).all()
assert (random_df[['n_train', 'n_val', 'n_test']].sum(axis=1) == len(model_df)).all()
print(f"baseline splits: train={random_df['n_train'].iloc[0]} "
      f"val={N_VAL_BASELINE} test={N_TEST_BASELINE}  (total {len(model_df)})")

random_df


## 4. Banpool protocol across 10 permutations of pool $\mathcal{B}$

Pool $\mathcal{A}$ (utterances containing **no** banlist word) is the fixed training set.
Pool $\mathcal{B}$ (utterances containing $\geq 1$ banlist word) is repeatedly partitioned 50/50
into validation and test. Only the $\mathcal{B}$ permutation varies with the seed — the training
set is invariant, which is why the Banpool standard deviations are tighter than the random
baseline's.


In [ ]:
banpool_runs = []
for seed in range(1, N_REPS + 1):
    tr, va, te = _quiet_banpool(model_df, seed)
    banpool_runs.append({'seed': seed, **measure(tr, va, te)})

banpool_df = pd.DataFrame(banpool_runs)

# sanity: pool A (train) must be identical across seeds
assert banpool_df['n_train'].nunique() == 1, 'pool A is not invariant across seeds!'
banpool_df


In [ ]:
# The specific deterministic split deployed in the experiments
tr42, va42, te42 = _quiet_banpool(model_df, DEPLOYED_SEED)
deployed = measure(tr42, va42, te42)

print(f'Banpool, seed_split = {DEPLOYED_SEED}')
print(f"  splits: train={deployed['n_train']}  val={deployed['n_val']}  test={deployed['n_test']}"
      f"  (total {deployed['n_train'] + deployed['n_val'] + deployed['n_test']})")
for k in ['val_word_oov', 'val_token_oov', 'test_word_oov', 'test_token_oov']:
    print(f'  {k:16s}: {deployed[k]:.2f}%')
print(f"\n  supporting: test word OOV by type      = {deployed['test_word_oov_type']:.2f}%")
print(f"              test word OOV by utterance = {deployed['test_word_oov_utt']:.2f}%")


## 5. Results table

Aggregates are $\mu \pm \sigma$ across the 10 repetitions (sample standard deviation, `ddof=1`).


In [ ]:
METRICS = ['val_word_oov', 'val_token_oov', 'test_word_oov', 'test_token_oov']

def agg(frame):
    return {m: (frame[m].mean(), frame[m].std(ddof=1)) for m in METRICS}

rand_agg, ban_agg = agg(random_df), agg(banpool_df)

summary = pd.DataFrame([
    {'protocol': f'Standard Random ({N_VAL_BASELINE} val / {N_TEST_BASELINE} test)',
     **{m: f'{rand_agg[m][0]:.1f} ± {rand_agg[m][1]:.1f}' for m in METRICS}},
    {'protocol': f"Banpool Protocol ({deployed['n_val']} val / {deployed['n_test']} test)",
     **{m: f'{ban_agg[m][0]:.1f} ± {ban_agg[m][1]:.1f}' for m in METRICS}},
    {'protocol': f'Banpool (seed_split = {DEPLOYED_SEED})',
     **{m: f'{deployed[m]:.1f}' for m in METRICS}},
])
summary


In [ ]:
ratio = ban_agg['test_word_oov'][0] / rand_agg['test_word_oov'][0]
print(f"Banpool raises test-set token-level word OOV by {ratio:.2f}x "
      f"({rand_agg['test_word_oov'][0]:.1f}% -> {ban_agg['test_word_oov'][0]:.1f}%)")
print(f"Token OOV is {ban_agg['test_token_oov'][0]:.1f}% under both protocols: the 39-token "
      f"inventory is fully covered by training under any partition.")

print(f"\nUnder Banpool, {deployed['test_word_oov_utt']:.0f}% of test utterances contain >= 1 unseen "
      f"word (by construction).")
rand_utt = random_df['test_word_oov_utt'].mean()
print(f"Under the random baseline, only {rand_utt:.1f}% do.")

# Where does the deployed seed sit inside the Banpool distribution?
print(f"\nPosition of seed_split = {DEPLOYED_SEED} within the Banpool distribution (z-scores):")
for m in ['val_word_oov', 'test_word_oov']:
    mu, sd = ban_agg[m]
    z = (deployed[m] - mu) / sd
    print(f"  {m:14s}: {deployed[m]:5.2f}%  vs  mu={mu:.2f} sd={sd:.2f}  ->  z = {z:+.2f} SD")

print("\n  NOTE: val and test OOV are mechanically anticorrelated -- pool B is split 50/50, so an\n"
      "  unseen-word utterance sent to val cannot also appear in test. A seed high on one is low\n"
      "  on the other. seed 42 is an off-centre draw, not a representative one.")


## 6. Emit the LaTeX table

Paste the output directly into the thesis. Values are generated from the runs above, so the
table can never drift from the code.


In [ ]:
def fmt_pm(t):
    mu, sd = t
    return f'${mu:.1f} \\pm {sd:.1f}$'

n_val_b, n_test_b = deployed['n_val'], deployed['n_test']

latex = rf"""\begin{{table}}[h]
\centering
\caption{{Token-level Out-of-Vocabulary (OOV) rates across splitting methodologies on $C_{{1,\text{{open}}}}$. The random baseline follows prior work in drawing {N_VAL_BASELINE} validation and {N_TEST_BASELINE} test utterances at random. Both aggregates report $\mu \pm \sigma$ across {N_REPS} random permutations.}}
\label{{tab:oov_analysis}}
\begin{{tabular}}{{lcccc}}
\toprule
\textbf{{Splitting Protocol}} & \textbf{{Val Word OOV (\%)}} & \textbf{{Val Token OOV (\%)}} & \textbf{{Test Word OOV (\%)}} & \textbf{{Test Token OOV (\%)}} \\
\midrule
Standard Random ({N_VAL_BASELINE}/{N_TEST_BASELINE}) & {fmt_pm(rand_agg['val_word_oov'])} & {fmt_pm(rand_agg['val_token_oov'])} & {fmt_pm(rand_agg['test_word_oov'])} & {fmt_pm(rand_agg['test_token_oov'])} \\
Banpool Protocol ({n_val_b}/{n_test_b}) & {fmt_pm(ban_agg['val_word_oov'])} & {fmt_pm(ban_agg['val_token_oov'])} & {fmt_pm(ban_agg['test_word_oov'])} & {fmt_pm(ban_agg['test_token_oov'])} \\
Banpool (\texttt{{seed\_split}} = {DEPLOYED_SEED}) & {deployed['val_word_oov']:.1f} & {deployed['val_token_oov']:.1f} & {deployed['test_word_oov']:.1f} & {deployed['test_token_oov']:.1f} \\
\bottomrule
\end{{tabular}}
\end{{table}}"""

print(latex)


In [ ]:
out_path = Path('oov_analysis_table.tex')
out_path.write_text(latex, encoding='utf-8')
print('wrote', out_path.resolve())


## 7. Interpretation

**The word branch.** The Banpool protocol roughly doubles the token-level word OOV rate of the
evaluation sets relative to a randomised split. Under a random partition the evaluation sets
inherit the corpus's Zipfian word distribution, so high-frequency types recur in training and the
measured error understates the model's behaviour on genuinely novel lexis. The Banpool split
removes that leakage by construction: every evaluation utterance contains at least one type held
out of training (100% utterance-level OOV, versus ~68% under the random baseline).

**The token branch.** Token OOV is 0.0% under *both* protocols. The 39-token inventory is
fully covered by the training split regardless of how utterances are partitioned, because
tokens recur across unrelated words. This is not a null result — it is the point. It shows the
Banpool protocol stresses the *lexical* generalisation axis specifically, while leaving the
phonetic label space fully observed. A token-level decoder therefore faces no unseen labels, and
any degradation it shows on the Banpool test set is attributable to sequence composition rather
than to unobserved output symbols. This is what licenses the word-vs-token comparison in RQ2.

**Variance.** Banpool standard deviations are tighter than the random baseline's because pool
$\mathcal{A}$ (training) is invariant across seeds; only the $\mathcal{B}$ permutation moves.

**Position of the deployed split.** `seed_split = 42` is *not* a central draw: it sits roughly
$+2.1\sigma$ above the Banpool mean on validation and $-2.0\sigma$ below it on test (z-scores
printed above). These two deviations are not independent. Pool $\mathcal{B}$ is partitioned 50/50,
so every unseen-word utterance assigned to validation is one that cannot land in test; the val and
test OOV rates are mechanically anticorrelated, and a seed that is high on one is low on the other.
The deployed split remains a valid draw — its test OOV (14.3%) still exceeds the random baseline's
mean test OOV (8.6%) by a wide margin, which is the claim the protocol needs to support — but it
should **not** be described as representative of the Banpool mean. The $\mu \pm \sigma$ rows exist
precisely so the reader can see where it falls.

**Scope.** These are lexical statistics of the transcripts. They characterise the partitioning
protocol, not model performance, and are computed on the silent parallel subset; because the
parallel modes share transcripts verbatim, the same numbers hold for the voiced parallel subset.
